**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 36: 프로젝트 배포

34번에서 학습한 LoRA 어댑터를 **실제 서비스로 배포**합니다.

### 흐름
```
1. LoRA 어댑터 로드 및 확인
2. FastAPI 서버 코드 생성 → 터미널에서 실행
3. API 테스트
4. Streamlit 챗봇 생성 → 터미널에서 실행
```


In [1]:
# =====================================================
# 📦 프로젝트 설정 및 어댑터 확인
# =====================================================
import json, os

OUTPUT_DIR = "./output/project"
ADAPTER_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")

with open(os.path.join(OUTPUT_DIR, "project_plan.json"), "r", encoding="utf-8") as f:
    project_plan = json.load(f)

MODEL_NAME = project_plan["model_config"]["base_model"]

# 어댑터 존재 확인
if os.path.exists(ADAPTER_PATH):
    size = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, fns in os.walk(ADAPTER_PATH) for f in fns
    )
    print(f"✅ LoRA 어댑터 확인: {ADAPTER_PATH}")
    print(f"   크기: {size / 1024 / 1024:.1f} MB")
    print(f"   모델: {MODEL_NAME}")
else:
    print("❌ 어댑터 없음 — 34번 노트북에서 학습을 먼저 실행하세요")


✅ LoRA 어댑터 확인: ./output/project/lora_adapter
   크기: 85.6 MB
   모델: Qwen/Qwen2.5-1.5B-Instruct


---
## 1️⃣ FastAPI 서버 생성

학습된 모델을 API로 서빙합니다.


In [2]:
# =====================================================
# 📝 FastAPI 서버 코드 생성
# =====================================================

server_code = f"""
# project_server.py - 파인튜닝 모델 API 서버
# 실행: python project_server.py

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import time
import os
import uvicorn

app = FastAPI(
    title="{project_plan['project_name']} API",
    description="{project_plan['task']}",
    version="1.0.0"
)

class ChatRequest(BaseModel):
    messages: List[dict]
    max_new_tokens: Optional[int] = 256
    temperature: Optional[float] = 0.7

class ChatResponse(BaseModel):
    response: str
    tokens_generated: int
    time_seconds: float

MODEL_NAME = "{MODEL_NAME}"
ADAPTER_PATH = os.path.join(os.path.dirname(os.path.abspath(__file__)), "lora_adapter")

@app.on_event("startup")
async def load_model():
    global model, tokenizer
    print(f"모델 로딩 중: {{MODEL_NAME}}")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    model.eval()
    print("모델 로딩 완료!")

@app.get("/health")
async def health():
    return {{"status": "healthy", "model": MODEL_NAME, "project": "{project_plan['project_name']}"}}

@app.post("/chat", response_model=ChatResponse)
async def chat(request: ChatRequest):
    try:
        text = tokenizer.apply_chat_template(
            request.messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        
        start = time.time()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=request.max_new_tokens,
                temperature=request.temperature,
                do_sample=request.temperature > 0,
            )
        elapsed = time.time() - start
        
        new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
        
        return ChatResponse(
            response=response_text,
            tokens_generated=len(new_tokens),
            time_seconds=round(elapsed, 2)
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=9200)
"""

server_path = os.path.join(OUTPUT_DIR, "project_server.py")
with open(server_path, "w", encoding="utf-8") as f:
    f.write(server_code)

print(f"✅ 서버 코드 생성: {server_path}")
print(f"\n🚀 실행 방법:")
print(f"   터미널에서: python {server_path}")
print(f"   API 문서: http://localhost:9200/docs")


✅ 서버 코드 생성: ./output/project/project_server.py

🚀 실행 방법:
   터미널에서: python ./output/project/project_server.py
   API 문서: http://localhost:9200/docs


---
## 2️⃣ API 테스트

위 서버를 터미널에서 실행한 후, 아래 셀로 테스트합니다.


In [7]:
# =====================================================
# 🧪 API 테스트
# ⚠️ 터미널에서 project_server.py가 실행 중이어야 합니다
# =====================================================
import requests

API_URL = "http://localhost:9200"

# 서버 상태 확인
try:
    health = requests.get(f"{API_URL}/health", timeout=5).json()
    print(f"✅ 서버 상태: {health}")
except:
    print("❌ 서버가 실행 중이 아닙니다.")
    print(f"   터미널에서: python {os.path.join(OUTPUT_DIR, 'project_server.py')}")

# 질문 테스트
test_prompts = project_plan["eval_prompts"][:3]

for prompt in test_prompts:
    try:
        response = requests.post(
            f"{API_URL}/chat",
            json={
                "messages": [
                    {"role": "system", "content": project_plan["data_config"]["system_prompt"]},
                    {"role": "user", "content": prompt}
                ],
                "max_new_tokens": 200,
            },
            timeout=60,
        )
        result = response.json()
        print(f"\n🔹 질문: {prompt}")
        print(f"   응답: {result['response'][:150]}")
        print(f"   시간: {result['time_seconds']}초, 토큰: {result['tokens_generated']}")
    except Exception as e:
        print(f"\n❌ 에러: {e}")


✅ 서버 상태: {'status': 'healthy', 'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'project': 'AI/ML 기술 Q&A sLLM'}

🔹 질문: LoRA 파인튜닝이 무엇인지 설명하세요.
   응답: LoRA(LoRA: Low-Rank Adaptation)는 대규모 언어 모델과 같은 복잡한 모델을 효율적으로 조정하는 방법입니다. 기존의 튜닝 방식에는 모든 파라미터를 업데이트하는 것이지만, LoRA는 모델의 특정 부분에 대해 저차원 행렬을 도입하여 가중치를 조정합니다
   시간: 11.82초, 토큰: 146

🔹 질문: GPU 메모리가 부족할 때 해결 방법을 알려주세요.
   응답: GPU 메모리가 부족할 경우, 먼저 사용 중인 작업을 줄이고 적절한 시점에서 메모리를 확장하는 것이 중요합니다. 예를 들어, 학습률을 감소시키거나 최적화 파라미터를 조정할 수 있습니다. 또한, 추가적인 메모리 리소스를 할당하거나 서브트래킹을 통해 이미지 처리 과정을 개
   시간: 8.7초, 토큰: 137

🔹 질문: 트랜스포머의 Attention 메커니즘을 쉽게 설명하세요.
   응답: 트랜스포머의 Attention 메커니즘은 입력 데이터의 각 요소 간의 관계를 모델링하는 방법입니다.想象一下，你正在一个人群中找某个人，Attention 메커니즘은 당신이 모든 사람의面孔을 동시에 보면서, 그人的特徴를 빠르게 파악할 수 있도록帮你搜索的过程와 같습니다. Att
   시간: 8.66초, 토큰: 140


---
## 3️⃣ Streamlit 챗봇 생성

API 서버와 연동하는 웹 채팅 인터페이스를 만듭니다.


In [8]:
# =====================================================
# 📝 Streamlit 챗봇 코드 생성
# =====================================================

streamlit_code = f"""
# project_chatbot.py - 파인튜닝 모델 챗봇
# 실행: streamlit run project_chatbot.py --server.port 9300

import streamlit as st
import requests

API_URL = "http://localhost:9200"
SYSTEM_PROMPT = "{project_plan['data_config']['system_prompt']}"

st.set_page_config(page_title="{project_plan['project_name']}", page_icon="🤖")
st.title("🤖 {project_plan['project_name']}")
st.caption("{project_plan['task']}")

# 채팅 이력
if "messages" not in st.session_state:
    st.session_state.messages = []

# 이전 메시지 표시
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# 사용자 입력
if prompt := st.chat_input("질문을 입력하세요..."):
    st.session_state.messages.append({{"role": "user", "content": prompt}})
    with st.chat_message("user"):
        st.write(prompt)
    
    # API 요청
    with st.chat_message("assistant"):
        with st.spinner("생각 중..."):
            try:
                messages = [
                    {{"role": "system", "content": SYSTEM_PROMPT}},
                ] + st.session_state.messages
                
                response = requests.post(
                    f"{{API_URL}}/chat",
                    json={{"messages": messages, "max_new_tokens": 300}},
                    timeout=60,
                )
                result = response.json()
                answer = result["response"]
            except Exception as e:
                answer = f"❌ 서버 연결 실패: {{e}}"
        
        st.write(answer)
        st.session_state.messages.append({{"role": "assistant", "content": answer}})

# 사이드바
with st.sidebar:
    st.markdown("### ℹ️ 정보")
    st.markdown(f"- 모델: `{MODEL_NAME}`")
    st.markdown(f"- 도메인: {project_plan['domain']}")
    if st.button("대화 초기화"):
        st.session_state.messages = []
        st.rerun()
"""

chatbot_path = os.path.join(OUTPUT_DIR, "project_chatbot.py")
with open(chatbot_path, "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print(f"✅ 챗봇 코드 생성: {{chatbot_path}}")
server_path = os.path.join(OUTPUT_DIR, 'project_server.py')
print(f"\n🚀 실행 방법:")
print(f"   1. 터미널 1: python {server_path}")
print(f"   2. 터미널 2: streamlit run {chatbot_path} --server.port 9300")
print(f"   3. 브라우저: http://localhost:9300")


✅ 챗봇 코드 생성: {chatbot_path}

🚀 실행 방법:
   1. 터미널 1: python ./output/project/project_server.py
   2. 터미널 2: streamlit run ./output/project/project_chatbot.py --server.port 9300
   3. 브라우저: http://localhost:9300


In [9]:
# =====================================================
# 🚀 Streamlit 챗봇 실행
# =====================================================
import subprocess

chatbot_path = os.path.join(OUTPUT_DIR, "project_chatbot.py")
cmd = f"streamlit run {chatbot_path} --server.port 9300"

print("🚀 아래 명령어를 터미널에서 실행하세요:")
print("="*60)
print(f"  {cmd}")
print("="*60)
print(f"\n브라우저에서 http://localhost:9300 접속")
print(f"⚠️ project_server.py(포트 9200)가 먼저 실행 중이어야 합니다")


🚀 아래 명령어를 터미널에서 실행하세요:
  streamlit run ./output/project/project_chatbot.py --server.port 9300

브라우저에서 http://localhost:9300 접속
⚠️ project_server.py(포트 9200)가 먼저 실행 중이어야 합니다


---
## 📝 프로젝트 최종 정리


In [10]:
# =====================================================
# 🎉 프로젝트 최종 요약
# =====================================================
print("🎉 프로젝트 완료!")
print("="*60)

files = {
    "프로젝트 계획": "project_plan.json",
    "학습 데이터": "train_data.json",
    "LoRA 어댑터": "lora_adapter/",
    "평가 결과": "eval_results.json",
    "API 서버": "project_server.py",
    "챗봇 앱": "project_chatbot.py",
}

for name, path in files.items():
    full = os.path.join(OUTPUT_DIR, path)
    exists = os.path.exists(full)
    print(f"  {'✅' if exists else '❌'} {name}: {full}")

print(f"\n📋 실행 방법:")
print(f"   터미널 1: python {os.path.join(OUTPUT_DIR, 'project_server.py')}")
print(f"   터미널 2: streamlit run {os.path.join(OUTPUT_DIR, 'project_chatbot.py')} --server.port 9300")
print(f"   브라우저: http://localhost:9300")


🎉 프로젝트 완료!
  ✅ 프로젝트 계획: ./output/project/project_plan.json
  ✅ 학습 데이터: ./output/project/train_data.json
  ✅ LoRA 어댑터: ./output/project/lora_adapter/
  ✅ 평가 결과: ./output/project/eval_results.json
  ✅ API 서버: ./output/project/project_server.py
  ✅ 챗봇 앱: ./output/project/project_chatbot.py

📋 실행 방법:
   터미널 1: python ./output/project/project_server.py
   터미널 2: streamlit run ./output/project/project_chatbot.py --server.port 9300
   브라우저: http://localhost:9300
